Contract details
-------------------

In [1]:
from ib_async import *

util.startLoop()

# util.logToConsole(logging.DEBUG)

ib = IB()
ib.connect("127.0.0.1", 7497, clientId=11)

<IB connected to 127.0.0.1:7497 clientId=11>

Suppose we want to find the contract details for AMD stock.
Let's create a stock object and request the details for it:

In [2]:
amd = Stock("AMD")

cds = ib.reqContractDetails(amd)

len(cds)

35

We get a long list of contract details. Lets print the first one:

In [3]:
cds[0]

ContractDetails(contract=Contract(secType='STK', conId=4391, symbol='AMD', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='AMD', tradingClass='NMS', comboLegs=[]), marketName='NMS', minTick=0.01, orderTypes='ACTIVETIM,ADJUST,ALERT,ALGO,ALLOC,AON,AVGCOST,BASKET,BENCHPX,COND,CONDORDER,DARKONLY,DARKPOLL,DAY,DEACT,DEACTDIS,DEACTEOD,DIS,GAT,GTC,GTD,GTT,HID,IBKRATS,ICE,IMB,IOC,LIT,LMT,LOC,MIDPX,MIT,MKT,MOC,MTL,NGCOMB,NODARK,NONALGO,OCA,OPG,OPGREROUT,PEGBENCH,POSTONLY,PREOPGRTH,REL,RPI,RTH,SCALE,SCALEODD,SCALERST,SNAPMID,SNAPMKT,SNAPREL,STP,STPLMT,SWEEP,TRAIL,TRAILLIT,TRAILLMT,TRAILMIT,WHATIF', validExchanges='SMART,AMEX,NYSE,CBOE,PHLX,ISE,CHX,ARCA,ISLAND,DRCTEDGE,BEX,BATS,EDGEA,CSFBALGO,JEFFALGO,BYX,IEX,EDGX,FOXRIVER,NYSENAT,PSX', priceMagnifier=1, underConId=0, longName='ADVANCED MICRO DEVICES', contractMonth='', industry='Technology', category='Semiconductors', subcategory='Electronic Compo-Semicon', timeZoneId='EST (Eastern Standard Time)', tradingHours='20191231:

The contract itself is in the 'contract' property of the contract details. Lets make a list of contracts and look at the first:

In [4]:
contracts = [cd.contract for cd in cds]

contracts[0]

Contract(secType='STK', conId=4391, symbol='AMD', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='AMD', tradingClass='NMS', comboLegs=[])

To better spot the difference between all the contracts it's handy to convert to a DataFrame. There is a utility function to do that:

In [ ]:
util.df(contracts)

We can see from this that AMD trades in different currencies on different exchanges.
Suppose we want the one in USD on the SMART exchange. The AMD contract is adjusted to
reflect that and becomes unique:

In [6]:
amd = Stock("AMD", "SMART", "USD")

assert len(ib.reqContractDetails(amd)) == 1

Lets try the same for Intel:

In [7]:
intc = Stock("INTC", "SMART", "USD")

assert len(ib.reqContractDetails(intc)) == 1

Let's try a non-existing contract:

In [8]:
xxx = Stock("XXX", "SMART", "USD")

assert len(ib.reqContractDetails(xxx)) == 0

Error 200, reqId 6: No security definition has been found for the request, contract: Stock(symbol='XXX', exchange='SMART', currency='USD', comboLegs=[])


or a Forex contract

In [9]:
eurusd = Forex("EURUSD")

assert len(ib.reqContractDetails(eurusd)) == 1

With the ``qualifyContracts`` method the extra information that is send back
from the contract details request is used to fill in the original contracts.

Lets do that with ``amd`` and compare before and aftwards:

In [10]:
amd

Stock(symbol='AMD', exchange='SMART', currency='USD', comboLegs=[])

In [11]:
ib.qualifyContracts(amd)
amd

Stock(conId=4391, symbol='AMD', exchange='SMART', primaryExchange='NASDAQ', currency='USD', localSymbol='AMD', tradingClass='NMS', comboLegs=[])

**TIP:** When printing a contract, the output can be copy-pasted and it will be valid Python code.

The ``conId`` that is returned can by itself be used to uniquely specify a contract:

In [12]:
contract_4391 = Contract(conId=4391)

ib.qualifyContracts(contract_4391)

assert contract_4391 == amd

A whole bunch of contracts can be qualified at the same time. A list of all the successfull ones is returned:

In [13]:
qualContracts = ib.qualifyContracts(amd, intc, xxx, eurusd)

assert intc in qualContracts
assert xxx not in qualContracts

Error 200, reqId 12: No security definition has been found for the request, contract: Stock(symbol='XXX', exchange='SMART', currency='USD', comboLegs=[])
Unknown contract: Stock(symbol='XXX', exchange='SMART', currency='USD', comboLegs=[])


There is also an API function to request stocks (only stocks) that match a pattern:

In [14]:
matches = ib.reqMatchingSymbols("intc")
matchContracts = [m.contract for m in matches]

matches

[ContractDescription(contract=Contract(secType='STK', conId=38709539, symbol='INTC', primaryExchange='MEXI', currency='MXN', comboLegs=[]), derivativeSecTypes=[]),
 ContractDescription(contract=Contract(secType='STK', conId=270639, symbol='INTC', primaryExchange='NASDAQ.NMS', currency='USD', comboLegs=[]), derivativeSecTypes=['CFD', 'OPT', 'IOPT', 'WAR', 'FUT']),
 ContractDescription(contract=Contract(secType='STK', conId=12178234, symbol='INTC', primaryExchange='EBS', currency='CHF', comboLegs=[]), derivativeSecTypes=[]),
 ContractDescription(contract=Contract(secType='STK', conId=11463493, symbol='INTCUSD', primaryExchange='EBS', currency='USD', comboLegs=[]), derivativeSecTypes=[]),
 ContractDescription(contract=Contract(secType='STK', conId=130861844, symbol='S', primaryExchange='MEXI', currency='MXN', comboLegs=[]), derivativeSecTypes=[]),
 ContractDescription(contract=Contract(secType='STK', conId=130861851, symbol='2S7', primaryExchange='FWB', currency='EUR', comboLegs=[]), deri

In [15]:
assert intc in matchContracts

In [16]:
ib.disconnect()